Przykładowy wynik

In [10]:
import utils_genetic as g

population_manager = g.PopulationManager(25, g.Maze(), 1)
population_manager.find_path()

Pokolenie nr: 0, najlepszy wynik: -24.68

--------------------
.........#...............
..#......#...............
.##......#...............
.#...#####...............
.#...**#.................
....S**#.................
.....*.#.................
.....*.#.................
##...**#.................
..#.#**#..#..............
..#.#***#.#..............
.....#***#...............
......#***..#............
...#...#**..#............
...#.#..**..####..###..##
...#..#.....#.#....#.....
####...######.#....#.....
..........#...#....#.....
..........#...######.....
....#.....#..............
....#.....#..............
....#.....#..........K...
....#.....#..............
....#....................
....#....................
--------------------
Pokolenie nr: 1, najlepszy wynik: -19.79

--------------------
.........#...............
..#......#...............
.##......#...............
.#...#####...............
.#...**#.................
...*S**#.................
..*****#.................
..*****#.............

Wyniki dla różnych wspolczynnikow mutacji

In [11]:
import statistics
import io
from contextlib import redirect_stdout

TEST_REPEAT = 25  #ilosc powtorzen dla danego testu
RATES_TO_TEST = [0.01, 0.05, 0.1, 0.2] #wspolczynniki mutacji

print(f"Rozpoczynam testy (każdy po {TEST_REPEAT} powtórzeń)...")

for rate in RATES_TO_TEST:
    results = []

    g.MUTATION_RATE = rate

    for i in range(TEST_REPEAT):
        maze = g.Maze()
        manager = g.PopulationManager(25, maze, 1) # populacja 25, elita 1

        f = io.StringIO()
        with redirect_stdout(f):
            manager.find_path()
        output = f.getvalue()

        if "CEL OSIAGNIETY" in output:
            parts = output.split("Pokolenie nr: ")
            last_part = parts[-1].split(",")[0]
            results.append(int(last_part))
        else:
            results.append(10000)

    #liczymy srednia
    avg = statistics.mean(results)
    print(f"Dla MUTATION_RATE = {g.MUTATION_RATE}: Średnia liczba pokoleń = {avg}")

Rozpoczynam testy (każdy po 25 powtórzeń)...
Dla MUTATION_RATE = 0.01: Średnia liczba pokoleń = 8077.2
Dla MUTATION_RATE = 0.05: Średnia liczba pokoleń = 6799.92
Dla MUTATION_RATE = 0.1: Średnia liczba pokoleń = 7893.04
Dla MUTATION_RATE = 0.2: Średnia liczba pokoleń = 10000


Wyniki dla różnych wielkości populacji

In [12]:
TEST_REPEAT = 25  #ilosc powtorzen dla danego testu
g.MUTATION_RATE = 0.05
POP_TO_TEST = [25, 50, 75, 100] #wielkosci populacji

print(f"Rozpoczynam testy (każdy po {TEST_REPEAT} powtórzeń)...")

for pop in POP_TO_TEST:
    results = []

    for i in range(TEST_REPEAT):
        maze = g.Maze()
        manager = g.PopulationManager(pop, maze, 1) #populacje do testowania

        f = io.StringIO()
        with redirect_stdout(f):
            manager.find_path()
        output = f.getvalue()

        if "CEL OSIAGNIETY" in output:
            parts = output.split("Pokolenie nr: ")
            last_part = parts[-1].split(",")[0]
            results.append(int(last_part))
        else:
            results.append(10000)

    #liczymy srednia
    avg = statistics.mean(results)
    print(f"Dla Populacji = {pop}: Średnia liczba pokoleń = {avg}")

Rozpoczynam testy (każdy po 25 powtórzeń)...
Dla Populacji = 25: Średnia liczba pokoleń = 6748.64
Dla Populacji = 50: Średnia liczba pokoleń = 1291.36
Dla Populacji = 75: Średnia liczba pokoleń = 318.16
Dla Populacji = 100: Średnia liczba pokoleń = 187.92


### Wyniki dla różnych parametrów elite variable

In [13]:
TEST_REPEAT = 25  # ilość powtórzeń
ELITES_TO_TEST = [0, 1, 2, 5, 10]  # różne wartości elite_variable

print(f"Rozpoczynam testy (każdy po {TEST_REPEAT} powtórzeń)...")

for elite in ELITES_TO_TEST:
    results = []

    for i in range(TEST_REPEAT):
        maze = g.Maze()
        manager = g.PopulationManager(25, maze, elite)  # populacja 25, różna elita

        f = io.StringIO()
        with redirect_stdout(f):
            manager.find_path()
        output = f.getvalue()

        if "CEL OSIAGNIETY" in output:
            parts = output.split("Pokolenie nr: ")
            last_part = parts[-1].split(",")[0]
            results.append(int(last_part))
        else:
            results.append(10000)

    avg = statistics.mean(results)
    print(f"Dla elite_variable = {elite}: Średnia liczba pokoleń = {avg}")


Rozpoczynam testy (każdy po 25 powtórzeń)...
Dla elite_variable = 0: Średnia liczba pokoleń = 5106.64
Dla elite_variable = 1: Średnia liczba pokoleń = 5302.2
Dla elite_variable = 2: Średnia liczba pokoleń = 7747.72
Dla elite_variable = 5: Średnia liczba pokoleń = 9257.76
Dla elite_variable = 10: Średnia liczba pokoleń = 9422.12


In [14]:
from utils_genetic import Traveler, Maze, PopulationManager

TEST_REPEAT = 25
POPULATION = 100 #najlepsza z testow
ELITE = 0 #najlepsza z testow

EVALUATES = [
    "distance_reward",
    "squared_distance",
    "two_phase"
]

print(f"Test evaluate (każdy po {TEST_REPEAT} powtórzeń)")

for eval_name in EVALUATES:
    Traveler.ACTIVE_EVALUATE = eval_name
    results = []
    successes = 0

    for i in range(TEST_REPEAT):
        maze = Maze()
        manager = PopulationManager(POPULATION, maze, ELITE)

        f = io.StringIO()
        with redirect_stdout(f):
            manager.find_path(use_ranking=True)
        output = f.getvalue()

        if "CEL OSIAGNIETY" in output:
            gen = int(output.split("Pokolenie nr: ")[-1].split(",")[0])
            results.append(gen)
            successes += 1
        else:
            results.append(10000)

    avg = statistics.mean(results)

    print(
        f"evaluate = {eval_name} | "
        f"Śr. pokoleń = {avg:.1f} | "
        f"Sukcesy = {successes}/{TEST_REPEAT}"
    )


Test evaluate (każdy po 25 powtórzeń)
evaluate = distance_reward | Śr. pokoleń = 258.6 | Sukcesy = 25/25
evaluate = squared_distance | Śr. pokoleń = 232.8 | Sukcesy = 25/25
evaluate = two_phase | Śr. pokoleń = 285.3 | Sukcesy = 25/25
